IMports and setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.config import DATA_DIR, TARGET_COLUMN, MODEL_RESULTS

# import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

from src.preprocessing_utils import (
    create_sequences,
    load_feature_engineered_dataset,
    parse_datetime_index,
    scale_train_test_data,
    split_features_and_target,
)

from src.model_utils import (
    create_lstm_model,
    evaluate_regression_model,
    get_early_stopping_callback,
    plot_actual_vs_predicted,
    plot_training_history,
    save_model
)

pd.set_option("display.max_columns", None)

#### Model Optimization

This notebook improves the previously developed models through hyperparameter tuning and controlled experimentation.  
The optimization focuses on Random Forest and LSTM models, followed by a comparison of performance before and after tuning.

Load datasets

In [ ]:
train_file_path = DATA_DIR / "train_feature_engineered_data.csv"
test_file_path = DATA_DIR / "test_feature_engineered_data.csv"

train_df = load_feature_engineered_dataset(train_file_path)
test_df = load_feature_engineered_dataset(test_file_path)

train_df = parse_datetime_index(train_df, datetime_column="date")
test_df = parse_datetime_index(test_df, datetime_column="date")

print("Training set shape:", train_df.shape)
print("Testing set shape:", test_df.shape)

Target and features split

In [ ]:
x_train_df, y_train_series = split_features_and_target(train_df, TARGET_COLUMN)
x_test_df, y_test_series = split_features_and_target(test_df, TARGET_COLUMN)

Scaling

In [ ]:
x_train_scaled, x_test_scaled, fitted_scaler = scale_train_test_data(
    x_train=x_train_df,
    x_test=x_test_df,
    scaler_type="standard",
)

print("Scaled X_train shape:", x_train_scaled.shape)
print("Scaled X_test shape:", x_test_scaled.shape)

## Optimization
#### Optimization Strategy

Two models were selected for optimization:

- Random Forest, because it achieved the best initial performance
- LSTM, because it is the main deep learning model and may improve with better tuning

The optimization experiments focus on improving generalization and reducing prediction error.

           .................

Random Forest - Optimization

In [ ]:
optimized_random_forest_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)

optimized_random_forest_model.fit(
    x_train_scaled,
    y_train_series.to_numpy(),
)

In [ ]:
# saving the Random Forest model
save_model(
    model=optimized_random_forest_model,
    file_name="optimized_random_forest_model.pkl",
)

In [ ]:
optimized_random_forest_predictions = optimized_random_forest_model.predict(x_test_scaled)

optimized_random_forest_metrics = evaluate_regression_model(
    y_true=y_test_series.to_numpy(),
    y_pred=optimized_random_forest_predictions,
)

optimized_random_forest_metrics

In [ ]:
# plotting predictions

plot_actual_vs_predicted(
    y_true=y_test_series.to_numpy()[:300],
    y_pred=optimized_random_forest_predictions[:300],
    title="optimized_random_forest_actual_vs_predicted",
)

- The Random Forest model was tuned by increasing the number of trees and constraining tree growth using depth and minimum sample parameters.  
- This helps reduce overfitting while preserving the model’s ability to capture non-linear relationships.

...........

LSTM

In [ ]:
optimized_sequence_length = 12

x_train_seq_opt, y_train_seq_opt = create_sequences(
    x_data=x_train_scaled,
    y_data=y_train_series.to_numpy(),
    sequence_length=optimized_sequence_length,
)

x_test_seq_opt, y_test_seq_opt = create_sequences(
    x_data=x_test_scaled,
    y_data=y_test_series.to_numpy(),
    sequence_length=optimized_sequence_length,
)

print("x_train_seq_opt shape:", x_train_seq_opt.shape)
print("y_train_seq_opt shape:", y_train_seq_opt.shape)
print("x_test_seq_opt shape:", x_test_seq_opt.shape)
print("y_test_seq_opt shape:", y_test_seq_opt.shape)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import SGD,Adam
from tensorflow.keras.callbacks import EarlyStopping

def create_optimized_lstm_model(input_shape):
    model = Sequential()

    model.add(LSTM(128, return_sequences=True, input_shape=input_shape))
    model.add(Dropout(0.2))

    model.add(LSTM(64))
    model.add(Dropout(0.2))

    model.add(Dense(1))

    return model


# Define optimizer
optimizer = Adam(learning_rate=0.0009)

# Create model
lstm_model_opt = create_optimized_lstm_model(
    input_shape=(x_train_seq_opt.shape[1], x_train_seq_opt.shape[2])
)

# Compile
lstm_model_opt.compile(
    optimizer=optimizer,
    loss="mse"
)

lstm_model_opt.summary()

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history_opt = lstm_model_opt.fit(
    x_train_seq_opt,
    y_train_seq_opt,
    validation_split=0.1,
    epochs=35,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
# saving the LSTM model
save_model(
    model=lstm_model_opt,
    file_name="optimized_lstm_model.pkl",
)

In [ ]:
# plotting training

plot_training_history(history_opt)

In [ ]:
# Predictions
y_pred_opt = lstm_model_opt.predict(x_test_seq_opt).flatten()

# Evaluate
from src.model_utils import evaluate_regression_model

metrics_opt = evaluate_regression_model(
    y_true=y_test_seq_opt,
    y_pred=y_pred_opt
)

metrics_opt

In [ ]:
from src.model_utils import plot_actual_vs_predicted

plot_actual_vs_predicted(
    y_true=y_test_seq_opt[:300],
    y_pred=y_pred_opt[:300],
    title="Optimized LSTM: Actual vs Predicted",
    file_name="optimized_lstm_actual_vs_predicted"
)

#### Comparison

In [ ]:
# results before optimization

previous_results_df = pd.DataFrame(
    [
        {"model": "Linear Regression", "MAE": 15.570595, "RMSE": 22.553753},
        {"model": "Random Forest", "MAE": 14.802757, "RMSE": 21.402411},
        {"model": "LSTM", "MAE": 16.567993, "RMSE": 25.496047},
    ]
)

previous_results_df

In [ ]:
# results after optimization

optimized_results_df = pd.DataFrame(
    [
        {
            "model": "Optimized Random Forest",
            "MAE": optimized_random_forest_metrics["MAE"],
            "RMSE": optimized_random_forest_metrics["RMSE"],
        },
        {
            "model": "Optimized LSTM",
            "MAE": optimized_lstm_metrics["MAE"],
            "RMSE": optimized_lstm_metrics["RMSE"],
        },
    ]
)

optimized_results_df

In [ ]:
# results altogether

all_results_df = pd.concat(
    [previous_results_df, optimized_results_df],
    ignore_index=True,
)

all_results_df

In [ ]:
# checks

all_results_df.sort_values(by="RMSE", ascending=True)

saving optimized model comparison

In [ ]:
optimization_results_path = MODEL_RESULTS / "optimization_results.csv"
all_results_df.to_csv(optimization_results_path, index=False)

print(f"Saved optimization results to: {optimization_results_path}")